In [2]:
import pandas as pd
import numpy as np
import torch
import os

print("🚀 启动顶会级图网络数据流水线 (Data Pipeline)...")

# ==========================================
# 1. 统一格式与读取数据
# ==========================================
def format_stkcd(code):
    """修复 CSMAR 吞掉的股票代码前导零"""
    return str(code).split('.')[0].zfill(6)

print("\n⏳ 正在加载与清洗原始数据 (已开启脏数据容错模式)...")

# 读取研发联盟 (提取发明人)
df_alliance = pd.read_csv("CA_EnterpriseRDAlliance.csv", encoding='utf-8', low_memory=False, on_bad_lines='skip')
df_alliance['symbol'] = df_alliance['symbol'].apply(format_stkcd)
df_alliance = df_alliance.dropna(subset=['inventor'])

# 读取违规标签 (重点修复了这里的报错)
df_violation = pd.read_csv("STK_Violation_Main.csv", encoding='utf-8', low_memory=False, on_bad_lines='skip')
df_violation['Symbol'] = df_violation['Symbol'].apply(format_stkcd)

# 读取财务特征
df_fin = pd.read_csv("FS_Comins(Merge Query).csv", encoding='utf-8', low_memory=False, on_bad_lines='skip')
df_fin['FS_Comins.Stkcd'] = df_fin['FS_Comins.Stkcd'].apply(format_stkcd)
df_fin = df_fin.drop_duplicates(subset=['FS_Comins.Stkcd'], keep='last')

# ==========================================
# 2. 构建超图的边 (Edge Index)
# ==========================================
print("\n🧬 正在解析复杂的 '发明人' 字符串，构建超图拓扑...")

# 真实的研发合作往往是 "张三;李四;王五" 这种字符串，我们需要把它炸开 (Explode)
df_alliance['inventor_list'] = df_alliance['inventor'].str.split(';')
df_exploded = df_alliance.explode('inventor_list')
df_exploded['inventor_list'] = df_exploded['inventor_list'].str.strip() # 去除两边空格

# 提取所有的 公司(超边) 和 科学家(节点)
unique_companies = df_exploded['symbol'].unique()
unique_inventors = df_exploded['inventor_list'].unique()

print(f"📊 发现 {len(unique_companies)} 家硬科技企业, {len(unique_inventors)} 名核心科学家。")

# 建立 ID 到 索引的映射字典
comp_to_idx = {code: i for i, code in enumerate(unique_companies)}
inv_to_idx = {name: i for i, name in enumerate(unique_inventors)}

# 生成 PyG 格式的 edge_index (第1行是科学家，第2行是公司)
src_nodes = df_exploded['inventor_list'].map(inv_to_idx).values
dst_edges = df_exploded['symbol'].map(comp_to_idx).values
edge_index = torch.tensor([src_nodes, dst_edges], dtype=torch.long)
print(f"✅ 成功生成超图关联矩阵 edge_index! 形状: {edge_index.shape}")

# ==========================================
# 3. 构建初始特征矩阵 (X)
# ==========================================
print("\n📈 正在对齐财务特征矩阵 (X)...")
# 我们只提取这几家存在于图谱中的公司特征
num_companies = len(unique_companies)
feature_cols = ['FS_Comins.B001101000', 'FS_Comins.B001216000', 'FS_Comins.B002000000', 'FS_Combas.A001000000', 'FS_Combas.A002000000']
num_features = len(feature_cols)

X = torch.zeros((num_companies, num_features))
for i, comp_code in enumerate(unique_companies):
    # 在财务表中查找该公司
    row = df_fin[df_fin['FS_Comins.Stkcd'] == comp_code]
    if not row.empty:
        # 提取财务数字并转化为 Float 张量 (遇到缺失值用 0 填充)
        vals = row[feature_cols].fillna(0).values[0]
        X[i] = torch.tensor(vals.astype(float), dtype=torch.float)

print(f"✅ 成功生成节点特征矩阵 X! 形状: {X.shape}")

# ==========================================
# 4. 构建暴雷标签矩阵 (Y)
# ==========================================
print("\n🎯 正在生成风控预测目标标签 (Y)...")
# 我们把在违规表里出现过的公司视为高风险 (1.0)，其余为安全 (0.0)
violators_set = set(df_violation['Symbol'].unique())

Y = torch.zeros((num_companies, 1))
for i, comp_code in enumerate(unique_companies):
    if comp_code in violators_set:
        Y[i, 0] = 1.0

# 统计一下正负样本比例
num_risky = int(Y.sum().item())
print(f"✅ 成功生成标签矩阵 Y! 形状: {Y.shape}")
print(f"⚠️ 风险样本统计: {num_risky} 家公司曾有违规记录, {num_companies - num_risky} 家健康存续。")

# ==========================================
# 5. 持久化保存！
# ==========================================
torch.save(edge_index, "graph_edge_index.pt")
torch.save(X, "node_features_X.pt")
torch.save(Y, "risk_labels_Y.pt")

print("\n🎉 伟大的一步！三大张量已全部保存为 .pt 文件。咱们随时可以开启图神经网络的终极训练！")

🚀 启动顶会级图网络数据流水线 (Data Pipeline)...

⏳ 正在加载与清洗原始数据 (已开启脏数据容错模式)...

🧬 正在解析复杂的 '发明人' 字符串，构建超图拓扑...
📊 发现 597 家硬科技企业, 76192 名核心科学家。


C:\Users\26841\AppData\Local\Temp\ipykernel_14800\484665208.py:54: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\torch\csrc\utils\tensor_new.cpp:256.)
  edge_index = torch.tensor([src_nodes, dst_edges], dtype=torch.long)


✅ 成功生成超图关联矩阵 edge_index! 形状: torch.Size([2, 1097582])

📈 正在对齐财务特征矩阵 (X)...
✅ 成功生成节点特征矩阵 X! 形状: torch.Size([597, 5])

🎯 正在生成风控预测目标标签 (Y)...
✅ 成功生成标签矩阵 Y! 形状: torch.Size([597, 1])
⚠️ 风险样本统计: 437 家公司曾有违规记录, 160 家健康存续。

🎉 伟大的一步！三大张量已全部保存为 .pt 文件。咱们随时可以开启图神经网络的终极训练！
